#**An Object Identifier** : This is a project on working on a classification model that may differenciate/identify objects in an image

I will be using huggingface(hf) throughout this project , it focuses more on finetuning a model rather than traing a model from absolute scratch due to hardware and time limitation



##importing necessary libraries

In [62]:
from datasets import load_dataset
from transformers import AutoImageProcessor
import torch, torch.nn as nn
from torchvision.transforms import (
    Compose,
    RandomResizedCrop,
    RandomHorizontalFlip,
    Resize,
    CenterCrop,
    ToTensor,
    Normalize
    )
from torch.utils.data import Dataset
from PIL import Image
import matplotlib.pyplot as plt


#login() into hf

In [1]:
from huggingface_hub import login, whoami

In [ ]:
login()

In [ ]:
whoami()

In [7]:
import torch
device='cuda' if torch.cuda.is_available else 'cpu'
print(device)

cuda


##loading data

**Dataset** : [ethz/food101 dataset](https://huggingface.co/datasets/ethz/food101), it consists of 75.8k train rows and 25.3k test rows

Food-101 was constructed to create a challenging fine-grained image classification benchmark for computer vision research. The 101 categories were chosen to represent dishes commonly photographed and shared on the social platform Foodspotting.com, with an intentional emphasis on visually distinct dishes to make category discrimination non-trivial.

In [37]:
df=load_dataset('ethz/food101',split='train[:100]') # split='train[:100]' only downloads the first 100 images

In [38]:
df

Dataset({
    features: ['image', 'label'],
    num_rows: 100
})

In [39]:
df=df.train_test_split(test_size=.2,seed=42)
print(df['train'][0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x7E886EA5AE90>, 'label': 6}


In [63]:
df

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 80
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 20
    })
})

In [43]:
labels=df['train'].features['label'].names
print(labels)

['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheesecake', 'cheese_plate', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras', 'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice', 'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich', 'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup', 'hot_dog', 'huevos_rancheros', 'hummus', 'ice_cream', 'lasagna', 'lobster_bisque', 'lobster_roll_sandwich', 'macaroni_and_cheese', 'macarons', 'miso_sou

In [44]:
id2label={i:name for i,name in enumerate(labels)}
label2id={name:i for i,name in enumerate(labels)}

##visualizing an image using PIL & matplotlib

In [78]:
df['train']

Dataset({
    features: ['image', 'label'],
    num_rows: 80
})

##Preprocessing & Augmentation

Image preprocessor : [google/vit-base-patch16-224-in21k](https://huggingface.co/google/vit-base-patch16-224-in21k)

The Vision Transformer (ViT) is a transformer encoder model (BERT-like) pretrained on a large collection of images in a supervised fashion, namely ImageNet-21k, at a resolution of 224x224 pixels.

In [52]:
processor=AutoImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

In [66]:
print(processor)

ViTImageProcessor {
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}



In [ ]:
size=processor.size['height'] # resize and ruturns the image as expected by the preprocessor
normalize=Normalize(mean=processor.image_mean,std=processor.image_std) # this creats a normalization function

'''
whats normalization ?
-> images have different values in [0,255] , normalization is a statistical fx that scales down the value to [0,1] or [-1,1] , making it better for a model to predict
'''

In [58]:
def preprocess_train(batch):
  batch['pixel_values']=[train_tf(img.convert('RGB')) for img in batch['images']]
  return batch
def preprocess_test(batch):
  batch['pixel_value']=[test_tf(img.convert('RGB')) for img in batch['images']]
  return batch

df['train'].set_transform(preprocess_train)
df['test'].set_transform(preprocess_test)

In [ ]:
'''class DATASET(Dataset):
  def __init__(self,df,transform=None):
    super().__init__()
    self.df=df
    self.transform=transform
  def __len__(self):
    return len(self.df)
  def __getitem__(self,index):
    item=self.dataset[index]
    image=item['image']
    label=item['label']
    image=image.convert('RGB')
    if self.transform:
      image=self.transform(image)
    return{
        'pixel_values':image,
        'labels':label
    }'''

In [64]:
df

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 80
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 20
    })
})